<a href="https://colab.research.google.com/github/samhoon000/Job-Portal-Database-Analysis/blob/main/Linkedin_Job_Posting_Cleaning_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset link -->  https://www.kaggle.com/datasets/arshkon/linkedin-job-postings

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df=pd.read_csv("postings.csv", engine='python',on_bad_lines='skip')

In [2]:
df.head()

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3769 entries, 0 to 3768
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   job_id                      3769 non-null   int64  
 1   company_name                3635 non-null   object 
 2   title                       3769 non-null   object 
 3   description                 3769 non-null   object 
 4   max_salary                  1078 non-null   float64
 5   pay_period                  1304 non-null   object 
 6   location                    3769 non-null   object 
 7   company_id                  3636 non-null   float64
 8   views                       3753 non-null   float64
 9   med_salary                  226 non-null    float64
 10  min_salary                  1078 non-null   float64
 11  formatted_work_type         3769 non-null   object 
 12  applies                     1228 non-null   float64
 13  original_listed_time        3769 

In [4]:
df=df.drop(columns=['job_posting_url','application_url','zip_code','fips','expiry','closed_time'])

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3769 entries, 0 to 3768
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   job_id                      3769 non-null   int64  
 1   company_name                3635 non-null   object 
 2   title                       3769 non-null   object 
 3   description                 3769 non-null   object 
 4   max_salary                  1078 non-null   float64
 5   pay_period                  1304 non-null   object 
 6   location                    3769 non-null   object 
 7   company_id                  3636 non-null   float64
 8   views                       3753 non-null   float64
 9   med_salary                  226 non-null    float64
 10  min_salary                  1078 non-null   float64
 11  formatted_work_type         3769 non-null   object 
 12  applies                     1228 non-null   float64
 13  original_listed_time        3769 

In [6]:
cols_to_drop = [
    'med_salary',        # ~95% missing
    'skills_desc',       # almost empty
    'remote_allowed',    # too sparse + not critical
    'currency',          # tied to salary (but salary already weak)
    'compensation_type', # sparse
    'pay_period',        # sparse
    'posting_domain',    #not useful for analysis
    'normalized_salary'  #~70% missing → unreliable
]

In [7]:
df = df.drop(columns=cols_to_drop)

In [8]:
df['views'] = df['views'].fillna(0)
df['applies'] = df['applies'].fillna(0)
df['min_salary'] = df['min_salary'].fillna(0)
df['max_salary'] = df['max_salary'].fillna(0)

In [9]:
df=df.dropna(subset=['company_name','description'])
df['company_id']=df['company_id'].fillna(-1)
df['formatted_experience_level'] = df['formatted_experience_level'].fillna('Not Specified')

In [10]:
df['listed_time'] = pd.to_datetime(df['listed_time'], unit='ms')

In [11]:
df['original_listed_time'] = pd.to_datetime(df['original_listed_time'], unit='ms')

In [12]:
df.head()

,job_id,company_name,title,description,max_salary,location,company_id,views,min_salary,formatted_work_type,applies,original_listed_time,application_type,formatted_experience_level,listed_time,sponsored,work_type
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,"Princeton, NJ",2774458.0,20.0,17.0,Full-time,2.0,2024-04-17 23:45:08,ComplexOnsiteApply,Not Specified,2024-04-17 23:45:08,0,FULL_TIME
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,"Cincinnati, OH",64896719.0,8.0,45000.0,Full-time,0.0,2024-04-16 14:26:54,ComplexOnsiteApply,Not Specified,2024-04-16 14:26:54,0,FULL_TIME
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,"New Hyde Park, NY",766262.0,16.0,140000.0,Full-time,0.0,2024-04-12 04:23:32,ComplexOnsiteApply,Not Specified,2024-04-12 04:23:32,0,FULL_TIME
5,91700727,Downtown Raleigh Alliance,Economic Development and Planning Intern,Job summary:The Economic Development & Plannin...,20.0,"Raleigh, NC",1481176.0,9.0,14.0,Internship,4.0,2024-04-18 16:01:39,ComplexOnsiteApply,Not Specified,2024-04-18 16:01:39,0,INTERNSHIP
6,103254301,Raw Cereal,Producer,Company DescriptionRaw Cereal is a creative de...,300000.0,United States,81942316.0,7.0,60000.0,Contract,1.0,2024-04-11 18:43:39,SimpleOnsiteApply,Not Specified,2024-04-11 18:43:39,0,CONTRACT


In [13]:
# df.to_csv("Job_Postings.csv")

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3635 entries, 0 to 3768
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   job_id                      3635 non-null   int64         
 1   company_name                3635 non-null   object        
 2   title                       3635 non-null   object        
 3   description                 3635 non-null   object        
 4   max_salary                  3635 non-null   float64       
 5   location                    3635 non-null   object        
 6   company_id                  3635 non-null   float64       
 7   views                       3635 non-null   float64       
 8   min_salary                  3635 non-null   float64       
 9   formatted_work_type         3635 non-null   object        
 10  applies                     3635 non-null   float64       
 11  original_listed_time        3635 non-null   datetime64[ns]
 1

In [15]:
companies=df[['company_id','company_name']]

In [16]:
companies.head()

,company_id,company_name
0,2774458.0,Corcoran Sawyer Smith
2,64896719.0,The National Exemplar
3,766262.0,"Abrams Fensterman, LLP"
5,1481176.0,Downtown Raleigh Alliance
6,81942316.0,Raw Cereal


In [17]:
jobs=df[[
    'job_id',
    'title',
    'company_id',
    'location',
    'formatted_experience_level',
    'work_type',
    'formatted_work_type',
    'application_type',
    'sponsored'
    ]].rename(columns={'formatted_experience_level':'experience_level'})

In [18]:
jobs.head()

,job_id,title,company_id,location,experience_level,work_type,formatted_work_type,application_type,sponsored
0,921716,Marketing Coordinator,2774458.0,"Princeton, NJ",Not Specified,FULL_TIME,Full-time,ComplexOnsiteApply,0
2,10998357,Assitant Restaurant Manager,64896719.0,"Cincinnati, OH",Not Specified,FULL_TIME,Full-time,ComplexOnsiteApply,0
3,23221523,Senior Elder Law / Trusts and Estates Associat...,766262.0,"New Hyde Park, NY",Not Specified,FULL_TIME,Full-time,ComplexOnsiteApply,0
5,91700727,Economic Development and Planning Intern,1481176.0,"Raleigh, NC",Not Specified,INTERNSHIP,Internship,ComplexOnsiteApply,0
6,103254301,Producer,81942316.0,United States,Not Specified,CONTRACT,Contract,SimpleOnsiteApply,0


In [36]:
job_metrics=df[['job_id','views','applies']]

In [20]:
job_metrics.head()

,job_id,views
0,921716,20.0
2,10998357,8.0
3,23221523,16.0
5,91700727,9.0
6,103254301,7.0


In [21]:
salaries=df[['job_id','min_salary','max_salary']]

In [22]:
salaries.head()

,job_id,min_salary,max_salary
0,921716,17.0,20.0
2,10998357,45000.0,65000.0
3,23221523,140000.0,175000.0
5,91700727,14.0,20.0
6,103254301,60000.0,300000.0


In [23]:
job_time=df[['job_id','listed_time','original_listed_time']]

In [24]:
job_time.head()

,job_id,listed_time,original_listed_time
0,921716,2024-04-17 23:45:08,2024-04-17 23:45:08
2,10998357,2024-04-16 14:26:54,2024-04-16 14:26:54
3,23221523,2024-04-12 04:23:32,2024-04-12 04:23:32
5,91700727,2024-04-18 16:01:39,2024-04-18 16:01:39
6,103254301,2024-04-11 18:43:39,2024-04-11 18:43:39


In [37]:
companies.to_csv('companies.csv',index=False,header=False)
jobs.to_csv('jobs.csv',index=False,header=False)
job_metrics.to_csv('job_metrics.csv',index=False,header=False)
salaries.to_csv('salaries.csv',index=False,header=False)
job_time.to_csv('job_time.csv',index=False,header=False)

In [31]:
companies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3635 entries, 0 to 3768
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   company_id    3635 non-null   float64
 1   company_name  3635 non-null   object 
dtypes: float64(1), object(1)
memory usage: 85.2+ KB


In [32]:
jobs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3635 entries, 0 to 3768
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   job_id               3635 non-null   int64  
 1   title                3635 non-null   object 
 2   company_id           3635 non-null   float64
 3   location             3635 non-null   object 
 4   experience_level     3635 non-null   object 
 5   work_type            3635 non-null   object 
 6   formatted_work_type  3635 non-null   object 
 7   application_type     3635 non-null   object 
 8   sponsored            3635 non-null   int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 284.0+ KB


In [33]:
job_metrics.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3635 entries, 0 to 3768
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   job_id  3635 non-null   int64  
 1   views   3635 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 85.2 KB


In [34]:
salaries.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3635 entries, 0 to 3768
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   job_id      3635 non-null   int64  
 1   min_salary  3635 non-null   float64
 2   max_salary  3635 non-null   float64
dtypes: float64(2), int64(1)
memory usage: 113.6 KB


In [35]:
job_time.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3635 entries, 0 to 3768
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   job_id                3635 non-null   int64         
 1   listed_time           3635 non-null   datetime64[ns]
 2   original_listed_time  3635 non-null   datetime64[ns]
dtypes: datetime64[ns](2), int64(1)
memory usage: 113.6 KB
